# Debugging, Evaluating and Monitoring LLM Applications with LangSmith

## Install OpenAI, and LangChain dependencies

Install the following httpx library version for compatibility with other libraries

In [ ]:
!pip install httpx==0.27.2

In [1]:
!pip install langchain==0.2.0
!pip install langchain-openai==0.1.7
!pip install langchain-community==0.2.0
!pip install langsmith==0.1.71

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 973.7/973.7 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.8/332.8 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.0/145.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.5/327.5 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.6/124.6 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: langsm

## Enter Open AI API Key

In [2]:
from getpass import getpass

OPENAI_KEY = getpass('Enter Open AI API Key: ')

Enter Open AI API Key: ··········


## Enter LangSmith API Key

In [3]:
from getpass import getpass

LANGSMITH_KEY = getpass('Enter LangSmith API Key: ')

Enter LangSmith API Key: ··········


## Setup Environment Variables

In [4]:
import os

os.environ['OPENAI_API_KEY'] = OPENAI_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
# you can change this based on specific apps and projects
os.environ["LANGCHAIN_PROJECT"] = f"My LLM App Project"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_KEY

# Debugging and Monitoring LLM Apps with LangSmith

We will look at various ways in which we can evaluate LLM App steps, also known as runs. Collection of runs make up a trace.

A Trace is essentially a series of steps that your application takes to go from input to output.

Each of these individual steps is represented by a Run.

A Project is simply a collection of traces.

![](https://i.imgur.com/hiMkTK9.png)



## Monitor LLM App Traces with LangSmith

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI


# Configure the chat prompt template and define the llm chain.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Act as a helpful AI Assistant"),
        ("human", "{human_input}"),
    ]
)

# Initialize the OpenAI Chat instance with specific model parameters.
chatgpt = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# create simple llm chain
llm_chain = (prompt
                |
             chatgpt
                |
             StrOutputParser()
)

In [6]:
prompt_txt = "Explain AI in 3 bullet points"
response = llm_chain.invoke({'human_input': prompt_txt})
print(response)

- AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, especially computer systems.
- It involves the development of algorithms and models that enable machines to perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making.
- AI technologies include machine learning, natural language processing, computer vision, and robotics, and are used in various applications across industries such as healthcare, finance, and transportation.


## Selective Tracing

In [7]:
os.environ["LANGCHAIN_TRACING_V2"] = "false"

In [8]:
# does not get traced anymore
prompt_txt = "Explain Generative AI in one line"
response = llm_chain.invoke({'human_input': prompt_txt})
print(response)

Generative AI is a type of artificial intelligence that can create new content, such as images, text, or music, based on patterns and data it has been trained on.


In [9]:
from langchain.callbacks.tracers import LangChainTracer

# You can configure a LangChainTracer instance to trace a specific invocation.
tracer = LangChainTracer()
prompt_txt = "Explain AI in one line"
response = llm_chain.invoke({'human_input': prompt_txt}, config={"callbacks": [tracer]})
print(response)

AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, typically computer systems.


In [10]:
# LangChain also supports a context manager for tracing a specific block of code.
from langchain_core.tracers.context import tracing_v2_enabled

prompt_txt = "Explain Generative AI in one line"
with tracing_v2_enabled():
    response = llm_chain.invoke({'human_input': prompt_txt})
    print(response)

Generative AI is a type of artificial intelligence that can create new content, such as images, text, or music, based on patterns and data it has been trained on.


## Log traces to specific projects dynamically

In [11]:
tracer = LangChainTracer(project_name='My LLM App Project - Test')
prompt_txt = "Explain AI in one line"
response = llm_chain.invoke({'human_input': prompt_txt}, config={"callbacks": [tracer]})
print(response)

AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, such as learning, reasoning, and problem-solving.


In [12]:
prompt_txt = "Explain Generative AI in one line"
with tracing_v2_enabled(project_name='My LLM App Project - Test'):
    response = llm_chain.invoke({'human_input': prompt_txt})
    print(response)

Generative AI is a type of artificial intelligence that can create new content, such as images, text, or music, based on patterns it has learned from existing data.


## Adding metadata and tags in traces

In [13]:
prompt_txt = "Explain Generative AI in one line"
with tracing_v2_enabled():
    response = llm_chain.invoke({'human_input': prompt_txt},  {"tags": ['AI', 'Data Science'],
                                                               "metadata": {"user": "dj", "team": "data science"}})
    print(response)

Generative AI is a type of artificial intelligence that can create new content, such as images, text, or music, based on patterns and data it has been trained on.


In [14]:
prompt_txt = "Explain which is the fastest animal?"
with tracing_v2_enabled():
    response = llm_chain.invoke({'human_input': prompt_txt},  {"tags": ['General Knowledge', 'Environment'],
                                                               "metadata": {"user": "bob", "team": "social science"}})
    print(response)

The fastest animal on land is the cheetah, which can reach speeds of up to 60-70 miles per hour (96-112 km/h) in short bursts covering distances up to 500 meters. In the air, the peregrine falcon holds the title for the fastest animal, reaching speeds of over 240 miles per hour (386 km/h) when diving to catch prey. In the water, the sailfish is considered the fastest swimmer, reaching speeds of up to 68 miles per hour (110 km/h).


## Customize run names

In [15]:
prompt_txt = "Explain what is Deep Learning?"
with tracing_v2_enabled():
    response = llm_chain.invoke({'human_input': prompt_txt},  {"tags": ['AI', 'Data Science'],
                                                               "metadata": {"user": "dj", "team": "data science"},
                                                               "run_name": "DJRunJuly2024_001"},
                                )
    print(response)

Deep learning is a subset of machine learning that involves training artificial neural networks to learn and make decisions from data. These neural networks are inspired by the structure and function of the human brain, with interconnected layers of nodes that process and transform data. 

Deep learning algorithms can automatically learn to identify patterns and features in large amounts of data, without the need for explicit programming. This makes deep learning particularly well-suited for tasks such as image and speech recognition, natural language processing, and other complex problems that involve processing and understanding large amounts of data.

Deep learning has seen significant advancements in recent years, leading to breakthroughs in various fields such as computer vision, speech recognition, and autonomous driving. It is a powerful tool for solving complex problems and is widely used in industry and research.


# Evaluating and Monitoring LLM Apps with LangSmith

Here we will create an evaluation dataset and then evaluate our LLM app using various metrics and see how we can monitor our application using LangSmith

In [16]:
os.environ['OPENAI_API_KEY'] = OPENAI_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
# you can change this based on specific apps and projects
os.environ["LANGCHAIN_PROJECT"] = f"My LLM App Project"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_KEY  # Update to your API key

## Access LLM App runs

In [17]:
from langsmith import Client

In [18]:
from datetime import datetime, timedelta

# Initialize a client
client = Client(timeout_ms=3600000)

todays_llm_runs = client.list_runs(
    project_name="My LLM App Project",
    start_time=datetime.now() - timedelta(days=1), # can change or remove this to retrieve more runs
    run_type="llm",
)

In [19]:
dataset = []
for run in todays_llm_runs:
    dataset.append((run.inputs, run.outputs))

In [20]:
dataset[0]

({'messages': [{'lc': 1,
    'type': 'constructor',
    'id': ['langchain', 'schema', 'messages', 'SystemMessage'],
    'kwargs': {'content': 'Act as a helpful AI Assistant', 'type': 'system'}},
   {'lc': 1,
    'type': 'constructor',
    'id': ['langchain', 'schema', 'messages', 'HumanMessage'],
    'kwargs': {'content': 'Explain what is Deep Learning?',
     'type': 'human'}}]},
 {'llm_output': {'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None},
  'run': None,
  'generations': [{'text': 'Deep learning is a subset of machine learning that involves training artificial neural networks to learn and make decisions from data. These neural networks are inspired by the structure and function of the human brain, with interconnected layers of nodes that process and transform data. \n\nDeep learning algorithms can automatically learn to identify patterns and features in large amounts of data, without the need for explicit programming. This makes deep learning particularly well-suited f

## Create an evaluation dataset of inputs and outputs

Ideally the outputs should be human-labeled or annotated outputs which are examples of ground-truth for input data. Here we will just use model outputs to quickly create an evaluation dataset

In [21]:
refined_dataset = []
for input, output in dataset:
    refined_dataset.append((input['messages'][1]['kwargs']['content'],
                            output['generations'][0]['text']))
refined_dataset = list(set(refined_dataset))
refined_dataset

[('Explain which is the fastest animal?',
  'The fastest animal on land is the cheetah, which can reach speeds of up to 60-70 miles per hour (96-112 km/h) in short bursts covering distances up to 500 meters. In the air, the peregrine falcon holds the title for the fastest animal, reaching speeds of over 240 miles per hour (386 km/h) when diving to catch prey. In the water, the sailfish is considered the fastest swimmer, reaching speeds of up to 68 miles per hour (110 km/h).'),
 ('Explain AI in 3 bullet points',
  '- AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, especially computer systems.\n- It involves the development of algorithms and models that enable machines to perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making.\n- AI technologies include machine learning, natural language processing, computer vision, and robotics, and are used in various applications across indus

In [22]:
refined_dataset

[('Explain which is the fastest animal?',
  'The fastest animal on land is the cheetah, which can reach speeds of up to 60-70 miles per hour (96-112 km/h) in short bursts covering distances up to 500 meters. In the air, the peregrine falcon holds the title for the fastest animal, reaching speeds of over 240 miles per hour (386 km/h) when diving to catch prey. In the water, the sailfish is considered the fastest swimmer, reaching speeds of up to 68 miles per hour (110 km/h).'),
 ('Explain AI in 3 bullet points',
  '- AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, especially computer systems.\n- It involves the development of algorithms and models that enable machines to perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making.\n- AI technologies include machine learning, natural language processing, computer vision, and robotics, and are used in various applications across indus

Let's also add in some human-labeled input-output examples where the output is created by humans

In [23]:
more_examples = [
  ("What is the largest mammal?", "The blue whale"),
  ("What do mammals and birds have in common?", "Both are homeothermic (warm-blooded) animals"),
  ("What's the main characteristic of amphibians?", "They live both in water and on land"),
]

for input, output in more_examples:
    refined_dataset.append((input, output))

refined_dataset

[('Explain which is the fastest animal?',
  'The fastest animal on land is the cheetah, which can reach speeds of up to 60-70 miles per hour (96-112 km/h) in short bursts covering distances up to 500 meters. In the air, the peregrine falcon holds the title for the fastest animal, reaching speeds of over 240 miles per hour (386 km/h) when diving to catch prey. In the water, the sailfish is considered the fastest swimmer, reaching speeds of up to 68 miles per hour (110 km/h).'),
 ('Explain AI in 3 bullet points',
  '- AI, or artificial intelligence, refers to the simulation of human intelligence processes by machines, especially computer systems.\n- It involves the development of algorithms and models that enable machines to perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making.\n- AI technologies include machine learning, natural language processing, computer vision, and robotics, and are used in various applications across indus

## Create an Evaluation Dataset in LangSmith

Here we will upload our dataset to LangSmith cloud and create our eval dataset

In [24]:
# Initialize a client
client = Client(timeout_ms=3600000)

# Storing inputs in a dataset lets us
# run chains and LLMs over a shared set of examples.
dataset = client.create_dataset(
    dataset_name='Sample LLM App Eval Dataset - DJTest001',
    description="Dataset of sample prompts and human outputs",
)
dataset

Dataset(name='Sample LLM App Eval Dataset - DJTest001', description='Dataset of sample prompts and human outputs', data_type=<DataType.kv: 'kv'>, id=UUID('960c0385-c888-4294-8c22-7c74e30ca181'), created_at=datetime.datetime(2024, 7, 1, 22, 16, 16, 152491, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2024, 7, 1, 22, 16, 16, 152491, tzinfo=datetime.timezone.utc), example_count=0, session_count=0, last_session_start_time=None)

In [26]:
for input_prompt, output_answer in refined_dataset:
    client.create_example(
        inputs={"question": input_prompt},
        outputs={"answer": output_answer},
        metadata={"source": "Wikipedia"},
        dataset_id=dataset.id,
    )

In [27]:
datasets = client.list_datasets()
list(datasets)

[Dataset(name='Sample LLM Dataset - DJTest001', description='Dataset of sample prompts and outputs', data_type=<DataType.kv: 'kv'>, id=UUID('5be2bf76-3cfd-40ae-93bf-83b69cfe30ca'), created_at=datetime.datetime(2024, 6, 5, 7, 27, 38, 300926, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2024, 6, 5, 7, 27, 38, 300926, tzinfo=datetime.timezone.utc), example_count=8, session_count=5, last_session_start_time=datetime.datetime(2024, 6, 5, 9, 23, 55, 134578)),
 Dataset(name='Sample LLM App Eval Dataset - DJTest001', description='Dataset of sample prompts and human outputs', data_type=<DataType.kv: 'kv'>, id=UUID('960c0385-c888-4294-8c22-7c74e30ca181'), created_at=datetime.datetime(2024, 7, 1, 22, 16, 16, 152491, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2024, 7, 1, 22, 16, 16, 152491, tzinfo=datetime.timezone.utc), example_count=8, session_count=0, last_session_start_time=None)]

## View evaluation dataset examples

You can get examples of data points from your eval dataset in the cloud anytime

In [28]:
examples = client.list_examples(dataset_name="Sample LLM App Eval Dataset - DJTest001")

In [29]:
for example in examples:
    print(example)

dataset_id=UUID('960c0385-c888-4294-8c22-7c74e30ca181') inputs={'question': "What's the main characteristic of amphibians?"} outputs={'answer': 'They live both in water and on land'} metadata={'source': 'Wikipedia'} id=UUID('ae9ae2ce-f861-4291-95e8-ed0dc736f652') created_at=datetime.datetime(2024, 7, 1, 22, 17, 22, 999479, tzinfo=datetime.timezone.utc) modified_at=datetime.datetime(2024, 7, 1, 22, 17, 22, 999479, tzinfo=datetime.timezone.utc) runs=[] source_run_id=None
dataset_id=UUID('960c0385-c888-4294-8c22-7c74e30ca181') inputs={'question': 'What do mammals and birds have in common?'} outputs={'answer': 'Both are homeothermic (warm-blooded) animals'} metadata={'source': 'Wikipedia'} id=UUID('c0cd6e61-a651-47dd-8816-aa4f0d23ec7e') created_at=datetime.datetime(2024, 7, 1, 22, 17, 22, 909460, tzinfo=datetime.timezone.utc) modified_at=datetime.datetime(2024, 7, 1, 22, 17, 22, 909460, tzinfo=datetime.timezone.utc) runs=[] source_run_id=None
dataset_id=UUID('960c0385-c888-4294-8c22-7c74e3

## Evaluate and Monitor LLM App performance

We will leverage various evaluation metrics from LangSmith to test our LLM App performance here

In [30]:
from langsmith.evaluation import evaluate

# Initialize a client
client = Client(timeout_ms=3600000)

results = evaluate(
    lambda x: llm_chain.invoke({'human_input' : x['question']}),
    client=client,
    data="Sample LLM App Eval Dataset - DJTest001",
    experiment_prefix="test_eval001",
)

View the evaluation results for experiment: 'test_eval001-582accec' at:
https://smith.langchain.com/o/a0ce8312-2910-55c9-8948-b4d123aa5d69/datasets/960c0385-c888-4294-8c22-7c74e30ca181/compare?selectedSessions=8b87501f-3595-41d7-897b-a52880436d4c




0it [00:00, ?it/s]

In [31]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa")

# Initialize a client
client = Client(timeout_ms=3600000)

results = evaluate(
    lambda x: llm_chain.invoke({'human_input' : x['question']}),
    client=client,
    data="Sample LLM App Eval Dataset - DJTest001",
    experiment_prefix="test_eval001",
    evaluators=[qa_evaluator]
)

View the evaluation results for experiment: 'test_eval001-3b683185' at:
https://smith.langchain.com/o/a0ce8312-2910-55c9-8948-b4d123aa5d69/datasets/960c0385-c888-4294-8c22-7c74e30ca181/compare?selectedSessions=3e8bb16b-33f6-4e11-b653-90b9cf73248c




0it [00:00, ?it/s]

In [32]:
correct_evaluator = LangChainStringEvaluator("labeled_criteria",
                                             config={ "criteria": "correctness"})
conciseness_evaluator =LangChainStringEvaluator("criteria",
                                                config={ "criteria": "conciseness"})
helpfulness_evaluator = LangChainStringEvaluator("criteria",
                                                 config={ "criteria": "helpfulness"})
semantic_evaluator = LangChainStringEvaluator("embedding_distance")

# Initialize a client
client = Client(timeout_ms=3600000)

results = evaluate(
    lambda x: llm_chain.invoke({'human_input' : x['question']}),
    client=client,
    data="Sample LLM App Eval Dataset - DJTest001",
    experiment_prefix="test_eval001",
    evaluators=[correct_evaluator, conciseness_evaluator,
                semantic_evaluator, helpfulness_evaluator]
)

View the evaluation results for experiment: 'test_eval001-f9021695' at:
https://smith.langchain.com/o/a0ce8312-2910-55c9-8948-b4d123aa5d69/datasets/960c0385-c888-4294-8c22-7c74e30ca181/compare?selectedSessions=fc33ce61-cf3b-495f-bef7-d20b7cd21d83




0it [00:00, ?it/s]